<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Decoder</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">El decoder toma representaciones previas y genera una salida secuencial paso a paso. Este notebook se centra en su papel cuando el modelo debe producir texto o continuar una secuencia.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Muchas tareas generativas dependen de que el modelo pueda construir la salida token a token de forma coherente.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender cómo el decoder usa contexto previo y representaciones disponibles para generar secuencias.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Generar texto no es emitir todo de una vez, sino decidir progresivamente cada siguiente token apoyándose en lo ya producido.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">El decoder combina atención sobre lo generado y, en arquitecturas encoder-decoder, también sobre la representación de entrada.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Definición del tensor de entrada del decoder
"""Este bloque define un tensor que representa una secuencia de embeddings correspondiente a la secuencia objetivo.
Este bloque importa "torch" para trabajar con tensores en PyTorch.

Aplicamos la función "torch.randn()" a la variable "secuencia_objetivo_embedded" para generar un tensor con valores aleatorios que simulan embeddings de tokens.
El primer argumento "1" indica el tamaño del batch.
El segundo argumento "5" indica la longitud de la secuencia de tokens.
El tercer argumento "16" indica la dimensión de los embeddings.
La variable "secuencia_objetivo_embedded" contiene vectores aleatorios que simulan embeddings de la secuencia objetivo.
Esta estructura permite aplicar atención autorregresiva dentro del decoder."""

import torch

secuencia_objetivo_embedded = torch.randn(1, 5, 16)

secuencia_objetivo_embedded # Muestra el tensor que simula los embeddings de la secuencia objetivo. 


In [ ]:
# Bloque 2 — Definición del tensor proveniente del encoder
"""Este bloque define un tensor que representa la salida del encoder.
Aplicamos la función "torch.randn()" a la variable "salida_encoder" para generar un tensor con valores aleatorios que simulan la salida del encoder.
El primer argumento "1" indica el tamaño del batch.
El segundo argumento "6" indica la longitud de la secuencia de tokens.
El tercer argumento "16" indica la dimensión de los embeddings.
La variable "salida_encoder" simula las representaciones contextuales generadas por el encoder.
Este tensor será utilizado en la atención cruzada del decoder."""

salida_encoder = torch.randn(1, 6, 16)

salida_encoder # Muestra el tensor que simula la salida del encoder. 


In [ ]:
# Bloque 3 — Definición de la atención autorregresiva enmascarada
"""Este bloque define la capa de atención autorregresiva del decoder.
Este bloque importa "torch.nn" para definir capas de redes neuronales en PyTorch.

Aplicamos la clase "nn.MultiheadAttention()" a la variable "atencion_autoregresiva" para definir una capa de atención autorregresiva.
El parámetro "embed_dim=" indica la dimensionalidad de los embeddings de entrada, en este caso, 16 dimensiones.
El parámetro "num_heads=" define el número de cabezas de atención, en este caso, 2 cabezas.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".
La variable "atencion_autoregresiva" almacena una instancia de la capa de atención configurada con los parámetros especificados.

La capa se utilizará con una máscara causal para que cada posición atienda únicamente a posiciones anteriores y a la actual."""

import torch.nn as nn

atencion_autoregresiva = nn.MultiheadAttention(embed_dim=16, num_heads=2, batch_first=True)

atencion_autoregresiva # Muestra la configuración de la capa de atención autorregresiva para el decoder.


In [ ]:
# Bloque 4 — Creación de la máscara causal
"""Este bloque crea una máscara causal que impide que cada token atienda a posiciones futuras.
Aplicamos el método ".size()" a la variable "secuencia_objetivo_embedded" para obtener la longitud de la secuencia objetivo.
El argumento "1" indica que queremos obtener el tamaño de la segunda dimensión de la secuencia objetivo.
La variable "longitud" almacena la longitud de la segunda dimensión de la secuencia objetivo.

Aplicamos la función "torch.ones()" a la variable "matriz" para crear una matriz de unos de tamaño (longitud x longitud).
Los argumentos "longitud, longitud" indican que queremos crear una matriz de unos con el mismo número de filas y columnas que la longitud de la secuencia objetivo.
La variable "matriz" contiene una matriz de unos de 5 filas y 5 columnas.

Aplicamos la función "torch.tril()" acompañada del método ".unsqueeze()" a la variable "matriz" para obtener una matriz triangular 
inferior con unos en las posiciones permitidas y ceros en las futuras.
El argumento "0" del método ".unsqueeze()" indica que queremos agregar una nueva dimensión al inicio de la matriz triangular 
inferior obtenida, convirtiéndola en un tensor de tres dimensiones.

La variable "mascara_causal" contiene un tensor de tres dimensiones que representa la máscara causal.
La comparación con cero del siguiente bloque convierte esta matriz en una máscara booleana que bloquea las posiciones futuras.""" 

longitud = secuencia_objetivo_embedded.size(1)

matriz = torch.ones(longitud, longitud)

mascara_causal = torch.tril(matriz).unsqueeze(0)

longitud, matriz, mascara_causal # Muestra la longitud de la secuencia objetivo, la matriz de unos y la máscara causal generada para la atención del decoder.


In [ ]:
# Bloque 5 — Aplicación de la atención autorregresiva
"""Este bloque aplica la atención autorregresiva sobre la secuencia objetivo.
Asignamos de forma múltiple el resultado de la llamada a la instancia de la clase a las variables "salida_autoregresiva" y "pesos_autoregresivos" 
mediante la expresión "atencion_autoregresiva(secuencia_objetivo_embedded, secuencia_objetivo_embedded, secuencia_objetivo_embedded,
attn_mask=(mascara_causal.squeeze(0) == 0))".

El argumento "secuencia_objetivo_embedded" es el conjunto de tensores densos de entrada que queremos procesar mediante la atención multicabeza.
Se pasa por triplicado para que la atención multicabeza pueda calcular las relaciones entre las consultas, claves y valores de la secuencia de entrada.

El parámetro "attn_mask=" recibe la máscara causal transformada para evitar que la atención considere posiciones futuras.
El método ".squeeze(0)" elimina la primera dimensión de la máscara y deja una máscara de dos dimensiones.
La comparación "== 0" convierte la máscara en booleana: "True" indica las posiciones futuras que deben bloquearse y "False", las posiciones permitidas.

La variable "salida_autoregresiva" contiene la representación contextualizada generada por la atención enmascarada.
La variable "pesos_autoregresivos" almacena los pesos de atención promediados entre cabezas por defecto.
Esta operación permite que cada token de la secuencia objetivo atienda únicamente a los tokens anteriores y al token actual, 
evitando el acceso a información futura."""

salida_autoregresiva, pesos_autoregresivos = atencion_autoregresiva(
    secuencia_objetivo_embedded,
    secuencia_objetivo_embedded,
    secuencia_objetivo_embedded,
    attn_mask=(mascara_causal.squeeze(0) == 0)
)

salida_autoregresiva, pesos_autoregresivos # Muestra la salida autorregresiva y los pesos de atención correspondientes.


In [ ]:
# Bloque 6 — Conexión residual y normalización
"""Este bloque aplica una conexión residual y normalización de capa a la salida autorregresiva.
Aplicamos la clase "nn.LayerNorm()" a la variable "capa_norm" para definir una capa de normalización de capa.
El argumento "16" indica la dimensionalidad de los embeddings de entrada.
La variable "capa_norm" almacena una instancia de la capa de normalización de capa.

Asignamos la llamada a la instancia de la clase a la variable "salida_norm_1" mediante la expresión "capa_norm(secuencia_objetivo_embedded + salida_autoregresiva)".
El argumento "secuencia_objetivo_embedded" representa la secuencia objetivo embebida.
El argumento "salida_autoregresiva" representa la salida de la capa autorregresiva.

Los valores se suman antes de aplicar la normalización de capa para mantener la información de la secuencia objetivo y la salida autorregresiva.
La variable "salida_norm_1" contiene la secuencia tras combinar la salida autorregresiva con la secuencia objetivo embebida y normalizarla."""

capa_norm = nn.LayerNorm(16)

salida_norm_1 = capa_norm(secuencia_objetivo_embedded + salida_autoregresiva)

salida_norm_1 # Muestra la salida tras la normalización de capa.


In [ ]:
# Bloque 7 — Definición de la atención cruzada
"""Este bloque define la capa de atención cruzada del decoder.
Aplicamos la clase "nn.MultiheadAttention()" a la variable "atencion_cruzada" para definir la capa de atención cruzada del decoder.
El parámetro "embed_dim=" indica la dimensionalidad de los embeddings de entrada, en este caso, 16 dimensiones.
El parámetro "num_heads=" define el número de cabezas de atención, en este caso, 2 cabezas.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".

La variable "atencion_cruzada" almacena una instancia de la capa que relaciona la secuencia objetivo con la salida del encoder.
Esta atención permite que el decoder incorpore información contextual proveniente de la entrada original."""

atencion_cruzada = nn.MultiheadAttention(embed_dim=16, num_heads=2, batch_first=True)

atencion_cruzada # Muestra la configuración de la capa de atención cruzada para el decoder.


In [ ]:
# Bloque 8 — Aplicación de la atención cruzada
"""Este bloque aplica la atención cruzada utilizando la secuencia objetivo como consultas y la salida del encoder como claves y valores.
Asignamos de forma múltiple la llamada a la instancia de la clase a las variables "salida_cruzada" y "pesos_cruzados" mediante la 
expresión "atencion_cruzada(salida_norm_1, salida_encoder, salida_encoder)".

El argumento "salida_norm_1" es el resultado de la normalización de los tensores de la secuencia objetivo.
El argumento "salida_encoder" simula la representación del encoder que se utilizará como claves y valores en la atención cruzada.
Este último argumento se pasa por duplicado para que la atención multicabeza pueda calcular las relaciones entre las consultas 
de la secuencia objetivo y las claves y valores de la salida simulada del encoder.

La variable "salida_cruzada" contiene la representación generada por la atención cruzada.
La variable "pesos_cruzados" almacena los pesos de atención promediados entre cabezas por defecto."""

salida_cruzada, pesos_cruzados = atencion_cruzada(salida_norm_1, salida_encoder, salida_encoder)

salida_cruzada, pesos_cruzados # Muestra la salida de la atención cruzada y los pesos de atención correspondientes.


In [ ]:
# Bloque 9 — Segunda conexión residual y normalización
"""Este bloque aplica una segunda conexión residual y normalización de capa.
Asignamos la llamada a la instancia de la clase a la variable "salida_norm_2" mediante la expresión "capa_norm(salida_norm_1 + salida_cruzada)".
El argumento "salida_norm_1" representa la salida de la primera normalización tras la atención autorregresiva.
El argumento "salida_cruzada" representa la salida de la atención cruzada.

Los valores se suman antes de aplicar la normalización de capa para combinar la información de la atención cruzada con la salida previa.
La variable "salida_norm_2" contiene la secuencia tras combinar la atención cruzada con la salida previa y normalizarla.
En este ejemplo se reutiliza la misma capa de normalización que en la conexión anterior."""

salida_norm_2 = capa_norm(salida_norm_1 + salida_cruzada)

salida_norm_2 # Muestra la salida normalizada tras la atención cruzada para la secuencia objetivo.


In [ ]:
# Bloque 10 — Red feed-forward del decoder
"""Este bloque define la red feed-forward del decoder.
Aplicamos la clase "nn.Sequential()" a la variable "feed_forward" para definir la red feed-forward del decoder.

El primer argumento es la llamada a la clase "nn.Linear()" y define la primera capa lineal de la red feed-forward.
En este caso, la dimensionalidad de la entrada es 16 y la de la salida de la capa lineal intermedia es 32.

El segundo argumento es la llamada a la clase "nn.ReLU()" e indica la función de activación que se aplicará tras la primera capa lineal.
La función de activación introduce no linealidad en la red, permitiendo que el modelo capture relaciones más complejas entre las características de entrada.

El tercer argumento es la llamada a la clase "nn.Linear()" y define la segunda capa lineal de la red feed-forward.
En este caso, la dimensionalidad de la entrada es 32 y la de la salida de la segunda capa lineal es 16, cerrando así la estructura de la red feed-forward del decoder.
La variable "feed_forward" almacena una instancia de la red feed-forward del decoder.

Asignamos la llamada a la instancia de la clase a la variable "salida_ff" mediante la expresión "feed_forward(salida_norm_2)".
El argumento "salida_norm_2" representa la secuencia de entrada normalizada tras la combinación con la salida de la atención del decoder.
La variable "salida_ff" contiene la salida de la red feed-forward del decoder aplicada a la secuencia normalizada."""

feed_forward = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, 16)
)

salida_ff = feed_forward(salida_norm_2)

salida_ff # Muestra la salida de la red feed-forward del decoder.


In [ ]:
# Bloque 11 — Tercera conexión residual y normalización
"""Este bloque aplica una tercera conexión residual y normalización de capa.
Asignamos la llamada a la instancia de la clase a la variable "salida_decoder" mediante la expresión "capa_norm(salida_norm_2 + salida_ff)".
El argumento "salida_norm_2" representa la salida de la segunda normalización tras la atención cruzada.
El argumento "salida_ff" representa la salida de la red feed-forward.

Los valores se suman antes de aplicar la normalización de capa para combinar la información de la red feed-forward con la salida previa.
La variable "salida_decoder" contiene la representación final generada por el decoder.
Esta salida incorpora atención autorregresiva, atención cruzada y refinamiento mediante la red feed-forward.
Se reutiliza la misma capa de normalización en las tres conexiones y no se añade codificación posicional ni se generan tokens."""

salida_decoder = capa_norm(salida_norm_2 + salida_ff)

salida_decoder # Muestra la representación final generada por el decoder para cada posición de la secuencia objetivo.


In [ ]:
# Bloque 12 — Resultados finales
"""Este bloque organiza los resultados del procesamiento del decoder.
La variable "resultado" almacena en forma de diccionario los diferentes componentes generados durante el procesamiento del decoder.
La estructura permite visualizar cada componente del mecanismo del decoder."""

resultado = {
    "salida_autoregresiva": salida_autoregresiva,
    "pesos_autoregresivos": pesos_autoregresivos,
    "salida_cruzada": salida_cruzada,
    "pesos_cruzados": pesos_cruzados,
    "salida_decoder": salida_decoder
}

resultado # Muestra el diccionario con los diferentes componentes generados durante el procesamiento del decoder.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo el decoder convierte representaciones y contexto previo en una salida secuencial generada paso a paso.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">La generación depende de mantener coherencia entre el historial producido y la siguiente decisión del modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Pensar la salida generativa como una etiqueta fija cuando su construcción es secuencial y dependiente del contexto.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Arquitectura Encoder-Decoder para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Arquitectura Encoder-Decoder y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">04_Transformers/05_La_arquitectura_encoder_decoder.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">